# Análisis exploratorio del corpus

In [ ]:
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import nltk
from nltk.corpus import stopwords
from spacy.lang.en.stop_words import STOP_WORDS

from whodunit_stylometry.utils.data_utils import get_file_inventory, read_book_text
from whodunit_stylometry.utils.nlp_utils import (
    compute_novel_metrics,
    compute_word_frecuencies,
    top_n_words,
    compute_ngrams_frecuencies,
    get_tokens_alpha_by_author,
)
from whodunit_stylometry.utils.stats_utils import add_iqr_outlier_flags
from whodunit_stylometry.utils.plot_utils import (
    plot_authors_pca,
    plot_novels_per_author,
    plot_publication_year_distribution,
    plot_metrics_histograms,
    plot_metrics_boxplots_by_author,
    plot_metrics_correlations_heatmap,
    plot_bars_by_author_with_std,
    plot_scatter_list_plotly,
    plot_standardized_heatmap_by_author,
    plot_zipf_curve_by_author,
)

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")

## Configuración

In [ ]:
CORPUS_DIR = Path("../corpus")
CORPUS_HC_DIR = Path("../corpus/hand_cleaned")
OUTPUT_DIR = Path("../output")
TABLES_DIR = OUTPUT_DIR / "tables"
FIGS_DIR = OUTPUT_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## Stopwords

Para el EDA voy a crear un conjunto de *stopwords* personalizado a partir de la unión de las *stopwords* de NLTK y las de spaCy obteniendo así un conjunto más completo. Esto es debido a que, aunque ambos conjuntos son bastante similares, cada uno tiene algunas palabras que el otro no incluye.

En principio no voy a sacar del conjunto de *stopwords* ninguna palabra como, por ejemplo, las negaciones, pero sí voy a añadir `"ain't"` que es una contracción común en inglés que no está incluida en ninguno de los dos conjuntos.

In [ ]:
# Obtengo las stopwords de NLTK para inglés
LANG = "english"
nltk_stopwords = set(stopwords.words(LANG))
print(f"Number of NLTK stopwords: {len(nltk_stopwords)}")
print(list(nltk_stopwords)[:20])

In [ ]:
# Obtengo las stopwords de spaCy para inglés y hago
# un pequeño reemplazo para normalizar las comillas
# simples tal y como haré luego con el texto del corpus
spacy_stopwords = {sw.replace("‘", "'").replace("’", "'") for sw in STOP_WORDS}
print(f"Number of spaCy stopwords: {len(spacy_stopwords)}")
print(list(spacy_stopwords)[:20])

In [ ]:
# Uno ambos conjuntos de stopwords para
# obtener un conjunto más completo
STOPWORDS = nltk_stopwords.union(spacy_stopwords)
print(f"Total of stopwords: {len(STOPWORDS)}")

In [ ]:
# Añado "ain't" al conjunto de stopwords
STOPWORDS.add("ain't")
print(f"Total of stopwords: {len(STOPWORDS)}")

In [ ]:
# sorted(list(STOPWORDS))

## Carga del corpus y sus metadatos

En primer luagr, vamos a cargar la información del listado de obras del corpus a partir del CSV de metadatos (`corpus_metadata.csv`).

A estos metadatos, les vamos a añadir una serie de información adicional a partir de los archivos de texto de las obras, como la ruta, el tamaño del archivo en bytes, su hash MD5 y el nombre normalizado del autor y del título de la obra.

De esta forma, las columnas del dataframe con la información del corpus quedarán de la siguiente forma:

- `file_name`: Nombre del archivo de texto de la obra.
- `author`: Nombre del autor de la obra.
- `author_norm`: Nombre normalizado del autor de la obra.
- `title`: Título de la obra.
- `title_norm`: Título normalizado de la obra, formado por una abreviatura de las iniciales del nombre del autor seguida de un guion y las palabras del título de la obra separadas por guiones bajos.
- `year`: Año de publicación de la obra.
- `type`: Tipo de obra (`novel` o `short stories`).
- `genre`: Género de la obra.
- `detective`: Nombre del detective protagonista de la obra, en caso de pertenecer a una serie de novelas protagonizadas por un mismo detective.
- `language`: Idioma de la obra. Todos los textos del corpus están en inglés.
- `country`: País de origen del autor de la obra.
- `source`: Repositorio de donde se ha obtenido el texto de la obra.
- `license`: Licencia de uso del texto de la obra.
- `url`: URL de donde se ha obtenido el texto de la obra.
- `path`: Ruta al archivo de texto de la obra en el proyecto.
- `size_bytes`: Tamaño del archivo de texto en bytes.
- `md5`: Hash MD5 del archivo de texto.

In [ ]:
# Cargo el inventario de obras del corpus
df_inv = get_file_inventory(CORPUS_HC_DIR)
print(df_inv.shape)
df_inv.head()

In [ ]:
# Cargo los metadatos del corpus
df_meta = pd.read_csv(CORPUS_DIR / "corpus_metadata.csv", sep=";")
print(df_meta.shape)
df_meta.head()

In [ ]:
assert len(df_inv) == len(df_meta), "The number of works in the inventory and in the metadata do not match."

In [ ]:
# Combino ambos dataframes para obtener un
# dataframe con toda la información del corpus
df_corpus = df_meta.merge(df_inv, on="file_name")
print(df_corpus.shape)
df_corpus.head()

In [ ]:
df_corpus.info()

## Resumen del corpus

In [ ]:
print("Number of authors:", df_corpus["author"].nunique())
print("Number of works:", df_corpus["title"].nunique())
print(f"Total corpus size: {round(df_corpus['size_bytes'].sum() / (1024**2), 2)} MB")
print(f"Temporal range of the corpus: {df_corpus['year'].min()} to {df_corpus['year'].max()}")

In [ ]:
plot_novels_per_author(df_corpus, FIGS_DIR / "works_by_author.png")

In [ ]:
plot_publication_year_distribution(df_corpus, year_col="year", save_path=FIGS_DIR / "publication_year_distribution.png")

In [ ]:
print("Distribution by genres:")
print(df_corpus["genre"].value_counts(dropna=False))

print("\nDistribution by types:")
print(df_corpus["type"].value_counts(dropna=False))

In [ ]:
print("Books that do not belong to a saga:", df_corpus["detective"].isna().sum())
print("\nDistribution by detectives:")
print(df_corpus["detective"].value_counts())

In [ ]:
print("Distribution by country of origin:")
print(df_corpus["country"].value_counts(dropna=False))

print("\nLanguage of the works:")
print(df_corpus["language"].value_counts(dropna=False))

In [ ]:
print("Distribution by repository:")
print(df_corpus["source"].value_counts(dropna=False))

In [ ]:
print("Distribution by author origin country:")
print(df_corpus["country"].value_counts(dropna=False))

## EDA del corpus

Vamos a calcular una serie de métricas para cada obra del corpus que nos permitarán caracterizarlas y compararlas con el resto de obras y hacer comparativas entre autores. Para ello usaremos los textos limpios de cada obra, es decir, los que hemos revisado manualmente para eliminar las anotaciones del *copyright*, índices, notas editoriales y otros elementos que no forman parte del texto de la obra en sí.

Para más información sobre la limpieza manual de los textos, su normalización y el detalle de la tokenización, consultar el apéndice correspondiente de la memoria.

Las métricas calculadas y su descripción son las siguientes:

- `n_tokens_all`: Número total de tokens del texto normalizado.
- `n_tokens_alpha`: Número de tokens alfabéticos del texto normalizado, es decir, aquellos que cumplen la expresión regular `ALPHA_TOKENS` definida en `nlp_utils.py`.
- `n_tokens_not_alpha`: Número de tokens no alfabéticos del texto normalizado, es decir, aquellos que no cumplen la expresión regular `ALPHA_TOKENS` definida en `nlp_utils.py`.
- `non_alpha_token_ratio`: Proporción de tokens no alfabéticos respecto al total de tokens del texto normalizado.
- `n_sentences`: Número de oraciones del texto normalizado.
- `avg_sentence_len`: Longitud media de las oraciones del texto normalizado.
- `median_sentence_len`: Mediana de la longitud de las oraciones del texto normalizado.
- `std_sentence_len`: Desviación estándar de la longitud de las oraciones del texto normalizado.Informa de si el texto mantiene un ritmo sintáctico regular o alterna frases muy cortas y muy largas.
- `sentences_per_1000_tokens`: Número de oraciones por cada 1000 tokens del texto normalizado.
- `short_sentence_ratio`: Proporción de oraciones cortas respecto al total de oraciones del texto normalizado. Se considera una oración corta aquella que tiene 10 o menos tokens alfabéticos.
- `long_sentence_ratio`: Proporción de oraciones largas respecto al total de oraciones del texto normalizado. Se considera una oración larga aquella que tiene 30 o más tokens alfabéticos.
- `n_paragraphs`: Número de párrafos del texto normalizado.
- `avg_paragraph_len`: Longitud media de los párrafos del texto normalizado.
- `median_paragraph_len`: Mediana de la longitud de los párrafos del texto normalizado.
- `paragraphs_per_1000_tokens`: Número de párrafos por cada 1000 tokens del texto normalizado.
- `std_paragraph_len`: Desviación estándar de la longitud de los párrafos del texto normalizado. Informa de si el texto mantiene una estructura de párrafos regular o alterna párrafos muy cortos y muy largos.
- `avg_sentences_per_paragraph`: Número medio de oraciones por párrafo del texto normalizado.
- `median_sentences_per_paragraph`: Mediana del número de oraciones por párrafo del texto normalizado.
- `n_chars`: Longitud total del texto normalizado en caracteres.
- `non_alpha_ratio`: Proporción de caracteres no alfabéticos respecto al total de caracteres del texto normalizado.
- `digit_ratio`: Proporción de caracteres dígitos respecto al total de caracteres del texto normalizado.
- `whitespace_ratio`: Proporción de caracteres de espacio en blanco respecto al total de caracteres del texto normalizado.
- `n_types`: Número de palabras alfabéticas distintas presentes en el texto.
- `ttr`: *Type-Token Ratio*, proporción entre el número de tipos y el número total de palabras alfabéticas.
- `hapax_count`: Número de palabras alfabéticas que aparecen una sola vez en el texto.
- `hapax_ratio`: Proporción de palabras alfabéticas únicas respecto al total de tipos.
- `avg_word_len`: Longitud media de las palabras alfabéticas del texto.
- `median_word_len`: Mediana de la longitud de las palabras alfabéticas del texto.
- `long_word_ratio`: Proporción de palabras largas respecto al total de palabras alfabéticas. Se considera una palabra larga aquella que tiene 8 o más caracteres alfabéticos.
- `mattr_100`: *Moving Average Type-Token Ratio* con una ventana de 100 tokens calcula la media de las proporciones de tipos respecto a tokens en ventanas móviles de 100 palabras alfabéticas a lo largo del texto.
- `stopword_ratio`: Proporción de palabras que son stopwords respecto al total de tokens de palabras alfabéticas.
- `comma_count`: Recuento total de comas en el texto normalizado.
- `period_count`: Recuento total de puntos en el texto normalizado.
- `semicolon_count`: Recuento total de puntos y coma en el texto normalizado.
- `colon_count`: Recuento total de dos puntos en el texto normalizado.
- `exclam_count`: Recuento total de signos de exclamación en el texto normalizado.
- `question_count`: Recuento total de signos de interrogación en el texto normalizado.
- `ellipsis_count`: Recuento total de puntos suspensivos en el texto normalizado.
- `quote_count`: Recuento total de comillas en el texto normalizado.
- `dialog_dash_count`: Recuento total de guiones de diálogo en el texto normalizado.
- `hyphen_count`: Recuento total de guiones en el texto normalizado.
- `parenthesis_count`: Recuento total de paréntesis en el texto normalizado.
- `punctuation_count_total`: Recuento total de signos de puntuación (comas, puntos, puntos y coma, dos puntos, signos de exclamación, signos de interrogación, puntos suspensivos, comillas, guiones de diálogo, guiones y paréntesis) en el texto normalizado.
- `punctuation_ratio`: Proporción de signos de puntuación respecto al total de caracteres del texto normalizado.
- `comma_count_per_1000`: Número de comas por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `period_count_per_1000`: Número de puntos por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `semicolon_count_per_1000`: Número de puntos y coma por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `colon_count_per_1000`: Número de dos puntos por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `exclam_count_per_1000`: Número de signos de exclamación por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `question_count_per_1000`: Número de signos de interrogación por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `ellipsis_count_per_1000`: Número de puntos suspensivos por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `quote_count_per_1000`: Número de comillas por cada 1000 tokens de palabras alfabéticas en el texto normalizado.
- `dialog_dash_count_per_1000`: Número de guiones de diálogo por cada 1000 tokens de palabras alfabéticas en el texto normalizado.

In [ ]:
# Cálculo de métricas para cada obra del corpus
# Este proceso puede tardar unos minutos, dependiendo
# del número de obras y su tamaño
metrics_list = []

for book in tqdm(df_corpus["path"]):
    text = read_book_text(Path(book))
    row_metrics = compute_novel_metrics(text, STOPWORDS)
    metrics_list.append(row_metrics)

metrics_df = pd.DataFrame(metrics_list)

df_corpus = pd.concat([df_corpus.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

In [ ]:
# Guardo el dataframe con las métricas calculadas
# para que no sea necesario volver a calcularlas cada
# vez que queramos hacer un análisis exploratorio del corpus
df_corpus.to_csv(TABLES_DIR / "corpus_eda_metrics.csv", index=False)

In [ ]:
# Cargamos las métricas pre-calculadas para cada obra del corpus
df_corpus = pd.read_csv(TABLES_DIR / "corpus_eda_metrics.csv")
print(df_corpus.shape)
df_corpus.head()

### Métricas de calidad de los textos del corpus

Estas métricas se calculan para evaluar la calidad superficial y la consistencia de cada texto antes del análisis estilométrico. En lugar de medir directamente el estilo literario, capturan propiedades básicas sobre la composición de los textos, como la proporción de caracteres no alfabéticos, dígitos y espacios en blanco, y permiten detectar posibles problemas de extracción, ruido del OCR, numeración de páginas, metadatos incrustados o diferencias en el formato editorial. Esto es importante porque un texto "sucio" o mal preprocesado puede introducir sesgos y hacer que aparentes diferencias estilísticas entre obras o autores sean en realidad artefactos del preprocesado.

La detección de *outliers* dentro del corpus para algunas de estas métricas, es decir, valores anómalos en `digit_ratio`, `non_alpha_ratio` o `whitespace_ratio`, pueden señalar obras que requieren una revisión manual o limpieza adicional antes de incorporarlas al análisis principal. Por tanto, funcionan como una capa de control de calidad del corpus, ayudando a garantizar que las comparaciones estilométricas se basen en rasgos lingüísticos reales y no en anomalías del texto.

In [ ]:
# Buscamos posibles outliers en las métricas relacionadas
# con la calidad de los textos del corpus
metric_cols = [
    "n_chars",
    "non_alpha_ratio",
    "digit_ratio",
    "whitespace_ratio",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)

In [ ]:
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "outlier_metrics"]]

Como vemos, varias novelas de Wilkie Collins aparecen como *outliers* en el corpus para el número de caracteres. Como sabemos, estas son obras largas y conocidas por su extensión, así que aquí `n_chars` no necesariamente señala un problema de calidad, sino una diferencia real de longitud.

Vamos a analizar más en detalle los *outliers* detectados para `digit_ratio` para ver si se trata de un problema de calidad del texto o si, por el contrario, es una característica real de la obra. Para ello, vamos a imprimir algunas muestras de las líneas de los textos que contienen dígitos para ver de este modo si se trata de numeración de páginas, metadatos incrustados o algún otro tipo de contenido que no forma parte del texto de la obra en sí.

In [ ]:
# Nos quedamos con los outliers detectados para `digit_ratio`
digit_outliers = df_checked[df_checked["outlier_metrics"].apply(lambda xs: "digit_ratio (high)" in xs)]
digit_outliers

digit_outliers[["file_name", "outlier_metrics", "path"]]

In [ ]:
# Imprimimos algunos fragmentos de los textos
# de los outliers detectados para `digit_ratio`
for path in digit_outliers["path"]:
    text = read_book_text(Path(path))
    lines_with_digits = [line for line in text.splitlines() if re.search(r"\d", line)]

    print("=" * 80)
    print(f"{path}\n")
    print("\n".join(random.sample(lines_with_digits, 20)))

No parece que haya un problema de calidad en estos textos, sino que se trata de una característica real de las obras, es decir, los dígitos forman parte del texto de la obra en sí, ya sea como parte de la narración o como parte de la estructura de la obra (por ejemplo, numeración de capítulos). Por tanto, no voy a eliminar estas obras del corpus ni a limpiar los dígitos de los textos, ya que esto podría eliminar información estilística relevante para el análisis estilométrico.

In [ ]:
metrics = [
    "n_chars",
    "non_alpha_ratio",
    "digit_ratio",
    "whitespace_ratio",
]

plot_metrics_histograms(df_corpus, metrics, save_path=FIGS_DIR / "quality_metrics_histograms.png")

In [ ]:
plot_metrics_boxplots_by_author(df_corpus, metrics, save_path=FIGS_DIR / "quality_metrics_boxplots.png")

In [ ]:
plot_bars_by_author_with_std(df_corpus, ["n_chars"], agg="mean", save_path=FIGS_DIR / "bars_by_author_with_std.png")

In [ ]:
# Mostramos una matriz de dispersión para visualizar
# las relaciones entre las métricas de calidad
sns.pairplot(df_corpus[metrics], corner=True, diag_kind="hist");

In [ ]:
plot_metrics_correlations_heatmap(df_corpus, metrics, save_path=FIGS_DIR / "quality_metrics_correlations_heatmap.png")

### Métricas de longitud y estructurales

Pasemos ahora a analizar las métricas relacionadas con la longitud y la estructura de los textos, donde variables como el número de tokens, la longitud media de las frases, la longitud media de los párrafos o el número total de frases y párrafos aportan información sobre la extensión de los textos y sobre su organización interna.

Estas medidas pueden considerarse rasgos estilométricos de nivel superficial, en la medida en que capturan propiedades formales del texto que, aun siendo relativamente simples, pueden contribuir a identificar diferencias de estilo entre autores o entre obras. Así, por ejemplo, una mayor longitud media de frase puede asociarse a una sintaxis más elaborada, mientras que la distribución de los párrafos puede ofrecer información sobre cómo el autor organiza su discurso.

In [ ]:
metric_cols = [
    "n_tokens_all",
    "n_tokens_alpha",
    "n_tokens_not_alpha",
    "non_alpha_token_ratio",
    "n_sentences",
    "avg_sentence_len",
    "median_sentence_len",
    "std_sentence_len",
    "sentences_per_1000_tokens",
    "short_sentence_ratio",
    "long_sentence_ratio",
    "n_paragraphs",
    "avg_paragraph_len",
    "median_paragraph_len",
    "std_paragraph_len",
    "paragraphs_per_1000_tokens",
    "avg_sentences_per_paragraph",
    "median_sentences_per_paragraph",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)

In [ ]:
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "outlier_metrics"]]

In [ ]:
plot_metrics_histograms(df_corpus, metric_cols, save_path=FIGS_DIR / "structural_metrics_histograms.png")

In [ ]:
plot_metrics_boxplots_by_author(
    df_corpus, metric_cols, save_path=FIGS_DIR / "structural_metrics_boxplots_by_author.png"
)

In [ ]:
plot_bars_by_author_with_std(df_corpus, metric_cols, agg="median", save_path=FIGS_DIR / "bars_by_author_with_std.png")

Ahora vamos a analizar una serie de pares de métricas parar ve cómo se correlacionan entre sí. Esto nos permitirá entender mejor las relaciones entre estas características estructurales y cómo se combinan para formar el estilo de cada obra.

* `("n_tokens_all", "n_sentences")`. Muestra cómo se relaciona la longitud total del texto con su segmentación en frases, y permite ver si los textos más largos crecen por acumular más frases.
* `("n_tokens_all", "n_paragraphs")`. Permite analizar si las obras más extensas también se estructuran en más párrafos o si, por el contrario, concentran más contenido en menos bloques.
* `("n_sentences", "n_paragraphs")`. Aporta información sobre el patrón de organización del texto, mostrando si un mayor número de frases suele traducirse en más párrafos.
* `("avg_sentence_len", "avg_paragraph_len")`. Permite observar si los textos con frases más largas tienden también a presentar párrafos más extensos, lo que puede reflejar un estilo más denso o desarrollado.
* `("avg_sentence_len", "median_sentence_len")`. Sirve para comprobar la estabilidad de la longitud de frase y detectar si la media está siendo arrastrada por frases excepcionalmente largas.
* `("avg_paragraph_len", "median_paragraph_len")`. Aporta la misma información anterior, pero a nivel de párrafo, permitiendo identificar asimetrías o valores extremos en la distribución.
* `("n_tokens_not_alpha", "n_tokens_all")`. Permite ver si la cantidad de tokens no alfabéticos crece simplemente con la longitud del texto o si algunos autores los usan proporcionalmente más.
* `("non_alpha_token_ratio", "avg_sentence_len")`. Ayuda a explorar si una mayor presencia relativa de signos, números o símbolos se asocia con frases más largas o con una puntuación más cargada.
* `("avg_sentences_per_paragraph", "avg_paragraph_len")`. Permite interpretar si los párrafos más largos lo son porque contienen más frases o porque esas frases son, además, más extensas.

In [ ]:
scatter_pairs = [
    ("n_tokens_all", "n_sentences"),
    ("n_tokens_all", "n_paragraphs"),
    ("n_sentences", "n_paragraphs"),
    ("avg_sentence_len", "avg_paragraph_len"),
    ("avg_sentence_len", "median_sentence_len"),
    ("avg_paragraph_len", "median_paragraph_len"),
    ("n_tokens_not_alpha", "n_tokens_all"),
    ("non_alpha_token_ratio", "avg_sentence_len"),
    ("avg_sentences_per_paragraph", "avg_paragraph_len"),
]

In [ ]:
plot_scatter_list_plotly(
    data=df_corpus,
    scatter_pairs=scatter_pairs,
    hue="author",
    work_col="title",
)

In [ ]:
plot_metrics_correlations_heatmap(
    df_corpus, metric_cols, save_path=FIGS_DIR / "structural_metrics_correlations_heatmap.png"
)

### Métricas de la estructura léxica de los textos del corpus

Estas métricas describen la estructura léxica básica de cada texto, es decir, cómo de amplio, variado y repetitivo es el vocabulario empleado y permiten caracterizar aspectos centrales del estilo de un autor.

`n_types` es el número de palabras distintas presentes en el texto. Si un texto contiene muchos tipos diferentes, su vocabulario es más amplio, si contiene pocos, es más repetitivo o más restringido.

`ttr` (*type-token ratio*) es la proporción entre el número de tipos y el número total de tokens. Esta métrica da una estimación de la diversidad léxica, cuanto más alta mayor variedad de vocabulario. Sin embargo, hay que interpretarla con cautela porque depende mucho de la longitud del texto y en textos largos suele bajar aunque el estilo no cambie.

`hapax_count` cuenta cuántas palabras aparecen una sola vez en el texto, y `hapax_ratio` expresa esas palabras únicas como proporción del número total de tipos. Estas métricas son útiles porque capturan el grado de rareza léxica. Un texto con muchos *hapax legomena* puede reflejar un vocabulario más variado, más específico o menos repetitivo. Aunque en estilometría, esto puede aportar señales sobre la riqueza expresiva del autor, también puede verse afectado por la longitud del texto o por el tema tratado.

Por último, `avg_word_len` y `median_word_len` calculan la longitud media y mediana de las palabras alfabéticas del texto. Estas medidas ofrecen aproximaciones simples al nivel de complejidad léxica. Aunque por sí solas no definen el estilo, pueden ayudar a distinguir entre textos con vocabulario más breve frente a otros con palabras más largas, cultas o específicas. De forma similar, `long_word_ratio` nos informa de la proporción de palabras largas respecto al total de palabras alfabéticas y se considera una palabra larga aquella que tiene 8 o más caracteres alfabéticos.

Para nuestro análisis, estas métricas son importantes porque capturan distintas dimensiones del uso del léxico: tamaño del vocabulario, diversidad, repetición, frecuencia de formas raras y complejidad superficial de las palabras.

In [ ]:
metric_cols = [
    "n_types",
    "ttr",
    "hapax_count",
    "hapax_ratio",
    "avg_word_len",
    "median_word_len",
    "long_word_ratio",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)

In [ ]:
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "n_chars", "type", "outlier_metrics"]]

Como veremos más adelenate, estos *outliers* léxicos no están señalando tanto problemas de calidad, sino sobre todo:

* efectos de la longitud de los textos
* diferencias entre novela y relato

Así, los *outliers* léxicos muestran un patrón consistente con las diferencias de longitud de los textos más que con errores de preprocesado. Los textos más breves tienden a presentar `ttr` alto y, en algunos casos, `hapax_ratio` alto, al tiempo que muestran valores bajos de `n_types` y `hapax_count`, lo que refleja un vocabulario más pequeño pero más diverso en proporción. 

En cambio, las novelas más largas tienden a concentrar valores altos de `n_types` y valores bajos de `hapax_ratio`, un comportamiento esperable dado que los textos extensos acumulan más tipos pero también más repetición léxica.

#### MATTR (*Moving-Average Type-Token Ratio*)

Se trata de una medida de diversidad léxica más estable que el TTR clásico ya que en vez de medir la diversidad léxica sobre todo el texto de una vez, la función la calcula sobre muchas ventanas deslizantes de tamaño fijo y luego promedia los resultados.

En cada ventana calcula:

$$\frac{\text{número de tokens distintos}}{\text{tamaño de la ventana}}$$

Luego guarda ese valor para todas las ventanas y al final devuelve la media de todos esos cocientes.

Mide cuánta variedad léxica hay de forma local y la promedia a lo largo del texto. Si en cada tramo de 100 palabras aparecen muchas palabras distintas, la MATTR será alta. Si en esos tramos hay mucha repetición, será baja.

Esta métrica corrige uno de los grandes problemas del TTR clásico que se calcula como:

$$\text{TTR} = \frac{\text{número de tipos}}{\text{número de tokens}}$$

El problema es que esta métrica depende mucho de la longitud del texto:

* en textos cortos suele salir artificialmente alta
* en textos largos suele salir artificialmente baja

Eso dificulta comparar novelas y relatos de distinta extensión.

La MATTR reduce ese sesgo porque siempre calcula la diversidad sobre ventanas del mismo tamaño. Así:

* hace las comparaciones más justas entre textos largos y cortos
* ofrece una medida más robusta de la riqueza léxica
* permite distinguir mejor entre diferencias de estilo reales y el efecto de longitud

En nuestro corpus, donde hay obras de distinta extensión, MATTR es especialmente útil porque ayuda a medir la diversidad léxica de manera más estable que:

* `ttr`
* `n_types`
* `hapax_count`
* `hapax_ratio`

Mientras que esas métricas están muy influenciadas por el tamaño del texto, MATTR suele reflejar mejor la variedad léxica efectiva del autor o de la obra.

A la hora de interpretar los valores:

* Una MATTR alta indica una mayor diversidad léxica, es decir, menos repetición dentro de cada ventana.
* Una MATTR baja refleja un vocabulario más repetitivo o más restringido.

En cuanto a que se considera “alto” o “bajo” no hay un umbral universal, lo importante es comparar entre textos del mismo corpus.

In [ ]:
metric_cols = [
    "mattr_100",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)

In [ ]:
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "n_chars", "type", "outlier_metrics"]]

La ausencia de outliers en MATTR sugiere que los outliers léxicos detectados anteriormente estaban motivados sobre todo por el efecto de la longitud de los textos y no por una diversidad léxica realmente anómala. Dado que MATTR calcula la diversidad léxica sobre ventanas deslizantes de tamaño fijo, es mucho menos sensible a la longitud global del texto que métricas como TTR, el número de tipos o la proporción de hapax. En este sentido, los resultados indican que la diversidad léxica del corpus es relativamente homogénea una vez controlado el efecto del tamaño.

Esto no significa que todos los textos sean iguales estilísticamente. Solo significa que, en términos de diversidad léxica medida robustamente, ninguno destaca de forma extrema.

#### Stopword ratio

Este cálculo es importante porque mide la proporción de palabras funcionales (*stopwords*) dentro de cada texto.

Las *stopwords* incluyen palabras muy frecuentes y gramaticales como artículos, preposiciones, pronombres o conjunciones, por ejemplo: *the, and, of, to, in, is*. Aunque suelen aportar poco contenido temático, sí reflejan bastante bien los hábitos de escritura. Precisamente por eso, en estilometría suelen ser útiles: tienden a depender menos del tema del texto y más de patrones relativamente estables del autor.

La métrica `stopword_ratio` calcula qué proporción de los tokens alfabéticos del texto pertenece al conjunto de *stopwords*. Si el valor es alto, significa que el texto contiene una mayor densidad de palabras funcionales, si es bajo, el texto está relativamente más cargado de palabras léxicas o de contenido.

Esta métrica es relevante por varias razones. Primero, ayuda a capturar una dimensión del estilo que no depende tanto del vocabulario del tema del que trata la obra como otras métricas léxicas. Segundo, puede revelar diferencias en la densidad gramatical o en el equilibrio entre palabras funcionales y palabras de contenido. Tercero, puede servir como indicador complementario para comparar obras y autores, especialmente cuando se interpreta junto con otras medidas como MATTR, la longitud media de palabra o métricas de puntuación.

In [ ]:
metric_cols = [
    "stopword_ratio",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "n_chars", "type", "outlier_metrics"]]

#### Visualización de las métricas léxicas

In [ ]:
metric_cols = [
    "n_types",
    "ttr",
    "hapax_count",
    "hapax_ratio",
    "avg_word_len",
    "median_word_len",
    "long_word_ratio",
    "mattr_100",
    "stopword_ratio",
]

In [ ]:
plot_metrics_histograms(df_corpus, metric_cols, save_path=FIGS_DIR / "lexical_metrics_histograms.png")

In [ ]:
plot_metrics_boxplots_by_author(df_corpus, metric_cols, save_path=FIGS_DIR / "lexical_metrics_boxplots_by_author.png")

In [ ]:
plot_bars_by_author_with_std(df_corpus, metric_cols, agg="median", save_path=FIGS_DIR / "bars_by_author_with_std.png")

Vamos a analizar varios pares de métricas léxicas para ver cómo se relacionan entre sí y si muestran patrones diferenciadores entre autores o entre novelas y relatos. En concreto, vamos a analizar los siguientes pares:

1. Dependencia con longitud del texto
  - `(n_tokens_all, n_types)`. Muestra cómo crece el vocabulario distinto con la longitud del texto (comportamiento tipo Heaps).
  - `(n_tokens_all, ttr)`. Evidencia el sesgo de TTR con la longitud: suele disminuir conforme el texto es más largo
  - `(n_tokens_all, mattr_100)`. Permite comprobar si MATTR (más robusta) es mucho menos dependiente del tamaño que TTR.
  - `(n_tokens_all, hapax_count)`. Indica si los hapax crecen casi linealmente con la longitud o si ciertos textos concentran más “raros” de lo esperado.
  - `(n_tokens_all, hapax_ratio)`. Mide cómo cambia la proporción de hapax con el tamaño, diferenciando diversidad real de efecto de longitud.
  - `(n_tokens_all, stopword_ratio)`. Ayuda a ver si textos más largos tienden a estabilizar su proporción de stopwords o si hay diferencias de registro/estilo.
 
2. Diversidad léxica (comparar medidas entre sí)
  - `(ttr, mattr_100)`. Contrasta una medida sensible a longitud (TTR) con una más estable (MATTR) para detectar efectos de tamaño y outliers.
  - `(n_types, ttr)`. Relaciona diversidad absoluta (tipos) con diversidad relativa (TTR) para ver si se comportan coherentemente.
  - `(n_types, mattr_100)`. Permite ver si textos con vocabulario más amplio también mantienen diversidad local alta en ventanas (MATTR).
  - `(hapax_ratio, ttr)`. Explora si mayor diversidad relativa viene acompañada de más palabras únicas, o si hay diversidad sin tantos hapax.
  - `(hapax_ratio, mattr_100)`. Relaciona rareza léxica (hapax) con diversidad local (MATTR) para distinguir “variedad sostenida” vs “picos de rareza”.
  - `(hapax_count, n_types)`. Mide qué parte del vocabulario distinto está compuesto por hapax y detecta textos con muchos tipos pero pocos hapax (o al revés).

3. Complejidad/longitud de palabra
  - `(avg_word_len, median_word_len)`. Sirve para detectar asimetrías: si la media se aleja de la mediana, hay cola de palabras muy largas.
  - `(avg_word_len, long_word_ratio)`. Comprueba si la media de longitud de palabra está explicada por una mayor proporción de palabras largas (consistencia interna).
  - `(median_word_len, long_word_ratio)`. Distingue si las palabras largas son frecuentes (sube la mediana) o solo una cola (sube ratio pero no tanto la mediana).
  - `(long_word_ratio, mattr_100)`. Explora si textos con palabras más largas también muestran más diversidad local, o si son “densos” pero repetitivos.
  - `(avg_word_len, mattr_100)`. Relaciona complejidad léxica superficial con diversidad léxica local para ver si van de la mano.
  - `(avg_word_len, ttr)`. Permite ver si estilos con palabras más largas tienden a mayor diversidad relativa (con la cautela del sesgo de TTR).
4. Stopwords como señal de registro/estilo
  - `(stopword_ratio, ttr)`. Muestra si textos con más palabras funcionales tienden a menor diversidad relativa (más repetición de función vs contenido).
  - `(stopword_ratio, mattr_100)`. Evalúa si una mayor carga funcional reduce la diversidad local de contenido de forma consistente.
  - `(stopword_ratio, hapax_ratio)`. Indica si textos más “informativos” (menos stopwords) tienden a introducir más vocabulario raro/único.
  - `(stopword_ratio, avg_word_len)`. Sirve como proxy de densidad de contenido: más stopwords suele asociarse a palabras más cortas y menor densidad léxica.
  - `(stopword_ratio, long_word_ratio)`. Contrasta estilo funcional vs léxico: menos stopwords suele ir con más palabras largas (más contenido), y viceversa.

In [ ]:
scatter_pairs = [
    # Length effects
    ("n_tokens_all", "n_types"),
    ("n_tokens_all", "ttr"),
    ("n_tokens_all", "mattr_100"),
    ("n_tokens_all", "hapax_count"),
    ("n_tokens_all", "hapax_ratio"),
    ("n_tokens_all", "stopword_ratio"),
    # Diversity metrics relationships
    ("ttr", "mattr_100"),
    ("n_types", "ttr"),
    ("n_types", "mattr_100"),
    ("hapax_ratio", "ttr"),
    ("hapax_ratio", "mattr_100"),
    ("hapax_count", "n_types"),
    # Word-length / “lexical complexity”
    ("avg_word_len", "median_word_len"),
    ("avg_word_len", "long_word_ratio"),
    ("median_word_len", "long_word_ratio"),
    ("long_word_ratio", "mattr_100"),
    ("avg_word_len", "mattr_100"),
    ("avg_word_len", "ttr"),
    # Stopwords as register/style signal
    ("stopword_ratio", "ttr"),
    ("stopword_ratio", "mattr_100"),
    ("stopword_ratio", "hapax_ratio"),
    ("stopword_ratio", "avg_word_len"),
    ("stopword_ratio", "long_word_ratio"),
]

In [ ]:
plot_scatter_list_plotly(
    data=df_corpus,
    scatter_pairs=scatter_pairs,
    hue="author",
    work_col="title",
)

In [ ]:
plot_metrics_correlations_heatmap(
    df_corpus, metric_cols, save_path=FIGS_DIR / "lexical_metrics_correlations_heatmap.png"
)

### Métricas de puntuación

En estilometría, las métricas de puntuación son útiles porque capturan rasgos formales del estilo que suelen ser relativamente estables y, en muchos casos, poco conscientes: la preferencia por comas frente a puntos, el uso de punto y coma o dos puntos, la frecuencia de paréntesis o rayas, o la presencia de signos de interrogación y exclamación. Estos patrones reflejan decisiones recurrentes sobre cómo el autor segmenta el discurso, introduce incisos, marca énfasis o gestiona el ritmo de la narración. Por ello, la puntuación actúa como una “huella” estilística complementaria al léxico, y puede contribuir a diferenciar autores incluso cuando los temas o el vocabulario varían.

Además, estas métricas permiten aproximarse a aspectos de estructura y complejidad sintáctica sin recurrir a análisis lingüísticos profundos: una mayor densidad de comas, incisos o signos como el punto y coma suele asociarse con oraciones más elaboradas o con mayor subordinación, mientras que la abundancia de puntos puede indicar un estilo más fragmentado o directo. También ayudan a caracterizar el registro y el género (por ejemplo, la puntuación asociada al diálogo mediante comillas o rayas) y a detectar posibles efectos editoriales o de normalización, ya que cambios de edición pueden alterar la puntuación y, con ello, el perfil estilométrico del texto.

In [ ]:
metric_cols = [
    "comma_count",
    "period_count",
    "semicolon_count",
    "colon_count",
    "exclam_count",
    "question_count",
    "ellipsis_count",
    "quote_count",
    "dialog_dash_count",
    "hyphen_count",
    "parenthesis_count",
    "punctuation_count_total",
    "punctuation_ratio",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "n_chars", "type", "outlier_metrics"]]

Además de los conteos absolutos, hemos calculado las frecuencias normalizadas de puntuación por cada 1.000 tokens alfabéticos. Esta normalización es importante porque los conteos brutos dependen mucho de la longitud del texto. Al expresar el uso de la puntuación como tasas, resulta posible comparar de forma más justa textos de distinto tamaño e interpretar estas variables como tendencias estilísticas y no solo como consecuencia de la extensión de la obra. Estas métricas ayudan a captar diferencias en ritmo sintáctico, expresividad y representación del diálogo dentro del corpus.

In [ ]:
metric_cols = [
    "comma_count_per_1000",
    "period_count_per_1000",
    "semicolon_count_per_1000",
    "colon_count_per_1000",
    "exclam_count_per_1000",
    "question_count_per_1000",
    "ellipsis_count_per_1000",
    "quote_count_per_1000",
    "dialog_dash_count_per_1000",
    "hyphen_count_per_1000",
    "parenthesis_count_per_1000",
]

df_checked, outlier_summary_df = add_iqr_outlier_flags(df_corpus, metric_cols)
outlier_summary_df

In [ ]:
df_checked[df_checked["has_any_outlier"]][["file_name", "n_chars", "type", "outlier_metrics"]]

#### Visualización de las métricas léxicas

In [ ]:
metric_cols = [
    "comma_count_per_1000",
    "period_count_per_1000",
    "semicolon_count_per_1000",
    "colon_count_per_1000",
    "exclam_count_per_1000",
    "question_count_per_1000",
    "ellipsis_count_per_1000",
    "quote_count_per_1000",
    "dialog_dash_count_per_1000",
    "hyphen_count_per_1000",
    "parenthesis_count_per_1000",
    "punctuation_ratio",
]

In [ ]:
plot_metrics_histograms(df_corpus, metric_cols, save_path=FIGS_DIR / "punctuation_metrics_histograms.png")

In [ ]:
plot_metrics_boxplots_by_author(
    df_corpus, metric_cols, save_path=FIGS_DIR / "punctuation_metrics_boxplots_by_author.png"
)

In [ ]:
plot_bars_by_author_with_std(df_corpus, metric_cols, agg="median", save_path=FIGS_DIR / "bars_by_author_with_std.png")

Los pares de métricas de puntuación que vamos a analizar son los siguientes:

- `(quote_count_per_1000, dialog_dash_count_per_1000)`. Distingue convenciones y grado de diálogo (comillas vs raya) y suele separar muy bien estilos/ediciones en narrativa.
- `(dialog_dash_count_per_1000, question_count_per_1000)`. En misterio las preguntas aparecen mucho en interrogatorios y diálogo, así que este par captura “intensidad interrogativa” asociada al discurso directo.
- `(dialog_dash_count_per_1000, exclam_count_per_1000)`. Refleja si el énfasis emocional (!), típico de tensión, se concentra en escenas dialogadas.
- `(ellipsis_count_per_1000, dialog_dash_count_per_1000)`. Relaciona pausas/suspense (…) con presencia de diálogo, muy propio de escenas de tensión o revelación.
- `(ellipsis_count_per_1000, question_count_per_1000)`. Mide si la suspensión/hesitación (…) coaparece con interrogación, señal de suspense e incertidumbre narrativa.
- `(exclam_count_per_1000, question_count_per_1000)`. Captura un eje de “intensidad” (¡?!) útil para diferenciar estilos más contenidas vs más expresivos.
- `(comma_count_per_1000, period_count_per_1000)`. Contrasta ritmo de prosa más fluida (comas) frente a uno más cortado (puntos), que en misterio puede asociarse a escenas rápidas.
- `(semicolon_count_per_1000, period_count_per_1000)`. Separa estilos más “clásicos/densos” (;) de estilos más directos y fragmentados (.), frecuente en thriller moderno.
- `(parenthesis_count_per_1000, dash_count_per_1000 / hyphen_count_per_1000)`. Compara dos formas de inciso/aclaración, útiles para ver narradores más digresivos vs más “cinemáticos” (incisos con raya/guion).
- `(punctuation_ratio, ellipsis_count_per_1000)`. Permite ver si la densidad global de puntuación está impulsada por marcas de suspense (…), y no solo por comas/puntos.

In [ ]:
scatter_pairs = [
    ("quote_count_per_1000", "dialog_dash_count_per_1000"),
    ("dialog_dash_count_per_1000", "question_count_per_1000"),
    ("dialog_dash_count_per_1000", "exclam_count_per_1000"),
    ("ellipsis_count_per_1000", "dialog_dash_count_per_1000"),
    ("ellipsis_count_per_1000", "question_count_per_1000"),
    ("exclam_count_per_1000", "question_count_per_1000"),
    ("comma_count_per_1000", "period_count_per_1000"),
    ("semicolon_count_per_1000", "period_count_per_1000"),
    ("parenthesis_count_per_1000", "hyphen_count_per_1000"),
    ("punctuation_ratio", "ellipsis_count_per_1000"),
]

In [ ]:
plot_scatter_list_plotly(
    data=df_corpus,
    scatter_pairs=scatter_pairs,
    hue="author",
    work_col="title",
)

In [ ]:
plot_metrics_correlations_heatmap(
    df_corpus, metric_cols, save_path=FIGS_DIR / "punctuation_metrics_correlations_heatmap.png"
)

### Métricas por autor

Vamos a analizar ahora las métricas estilométricas agrupadas por autor para ver si se detectan patrones diferenciadores entre ellos. Para ello, vamos a calcular los estadísticos descriptivos de una serie de métricas para cada autor y luego vamos a visualizar las medias para comparar entre autores.

En este caso, nos vamos a centrar solo en aquellas métricas más estables y robustas que permiten hacer comparaciones justas entre obras de distinta extensión, es decir, aquellas métricas que no están tan influenciadas por la longitud del texto y que, por tanto, pueden reflejar diferencias estilísticas reales entre autores sin estar sesgadas por el tamaño de las obras. En concreto, nos vamos a centrar en métricas como `mattr_100`, `stopword_ratio` o las métricas de puntuación normalizadas por cada 1.000 tokens alfabéticos.

In [ ]:
stable_metrics = [
    # structural
    "avg_sentence_len",
    "median_sentence_len",
    "std_sentence_len",
    "sentences_per_1000_tokens",
    "short_sentence_ratio",
    "long_sentence_ratio",
    "avg_paragraph_len",
    "median_paragraph_len",
    "std_paragraph_len",
    "paragraphs_per_1000_tokens",
    "avg_sentences_per_paragraph",
    "median_sentences_per_paragraph",
    # lexical
    "mattr_100",
    "stopword_ratio",
    "avg_word_len",
    "median_word_len",
    "long_word_ratio",
    "non_alpha_token_ratio",
    # punctuation
    "punctuation_ratio",
    "comma_count_per_1000",
    "period_count_per_1000",
    "semicolon_count_per_1000",
    "colon_count_per_1000",
    "exclam_count_per_1000",
    "question_count_per_1000",
    "ellipsis_count_per_1000",
    "quote_count_per_1000",
    "dialog_dash_count_per_1000",
    "hyphen_count_per_1000",
    "parenthesis_count_per_1000",
]

In [ ]:
# Obtenemos un dataframe con los estadísticos descriptivos
# de las métricas seleccionadas para cada autor
desc_by_author = df_corpus.groupby("author")[stable_metrics].describe()

desc_by_author.columns = [
    f"{metric}_{stat}".replace("25%", "p25").replace("50%", "p50").replace("75%", "p75")
    for metric, stat in desc_by_author.columns
]

desc_by_author = desc_by_author.reset_index()
print(desc_by_author.shape)
desc_by_author.head()

El heatmap de métricas estandarizadas por autor que mostramos a continuación nos ofrece una visión global y comparativa de los perfiles estilísticos de los autores. Cada fila representa un autor, cada columna una métrica, y el color indica si ese autor está por encima o por debajo de la media del conjunto en ese rasgo concreto. Al estandarizar las métricas por columna, el gráfico no se centra en los valores absolutos, sino en las diferencias relativas entre autores, lo que permite identificar de un vistazo qué rasgos caracterizan más a cada uno.

Este tipo de visualización resulta especialmente útil para detectar patrones generales, similitudes entre autores y métricas que discriminan mejor entre estilos. Permite leer cada fila como una especie de “huella estilística”, donde se combinan aspectos de sintaxis, léxico o puntuación. Sin embargo, conviene interpretarlo como una herramienta exploratoria: muestra tendencias relativas, pero no sustituye a gráficos más específicos como barplots o boxplots cuando se quiere analizar en detalle una métrica concreta o la distribución real de los valores.

In [ ]:
plot_standardized_heatmap_by_author(
    desc_by_author, value_cols="_mean", save_path=FIGS_DIR / "standardized_metrics_heatmap_by_author.png"
)

El análisis de PCA permite reducir el conjunto de métricas estilísticas a un espacio de pocas dimensiones, normalmente dos componentes principales, para visualizar de forma sintética las similitudes y diferencias entre autores. En este tipo de gráfico, cada punto representa un autor y su posición resume el comportamiento conjunto de muchas variables a la vez. Los autores que aparecen cercanos tienden a compartir perfiles estilísticos parecidos según las métricas consideradas, mientras que los que quedan más alejados presentan diferencias más marcadas.

La principal utilidad del PCA es ofrecer una visión estructural del conjunto de autores, facilitando la detección de agrupamientos, separaciones y patrones globales que no siempre son evidentes al observar métricas individuales por separado. Además, el análisis de las cargas de cada componente permite interpretar qué variables contribuyen más a esas diferencias, ayudando a entender si la separación entre autores está más relacionada con aspectos sintácticos, léxicos o de puntuación. Como ocurre con el heatmap, se trata de una herramienta exploratoria: resume mucha información de manera muy eficaz, pero conviene complementarla con visualizaciones más concretas para interpretar en detalle los rasgos que explican la posición de cada autor.

In [ ]:
mean_cols = [c for c in desc_by_author.columns if c.endswith("_mean")]
X = desc_by_author.set_index("author")[mean_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=X.index).reset_index()

plot_authors_pca(pca, pca_df, save_path=FIGS_DIR / "authors_pca.png")

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=X.columns, columns=["PC1", "PC2"])

print("Top métricas en PC1")
print(loadings["PC1"].sort_values(key=np.abs, ascending=False).head(10))

print("\nTop métricas en PC2")
print(loadings["PC2"].sort_values(key=np.abs, ascending=False).head(10))

- `avg_sentence_len_mean vs comma_count_per_1000_mean`: ayuda a ver si los autores con frases más largas también tienden a usar más comas, lo que suele reflejar una sintaxis más encadenada o subordinada.
- `avg_sentence_len_mean vs short_sentence_ratio_mean`: permite contrastar directamente estilos de frase larga frente a estilos más fragmentados o más cortantes.
- `avg_sentence_len_mean vs long_sentence_ratio_mean`: muestra de forma muy limpia qué autores no solo tienen frases largas en promedio, sino que además recurren con frecuencia a estructuras extensas.
- `avg_paragraph_len_mean vs avg_sentences_per_paragraph_mean`: sirve para distinguir autores con párrafos largos por acumulación de oraciones frente a autores con oraciones más largas dentro de párrafos no necesariamente extensos.
- `mattr_100_mean vs avg_word_len_mean`: combina diversidad léxica y longitud media de palabra, muy útil para detectar estilos más ricos, densos o elaborados a nivel vocabular.
- `mattr_100_mean vs stopword_ratio_mean`: permite explorar la oposición entre diversidad léxica y dependencia de palabras funcionales, que suele separar estilos más informativos de estilos más apoyados en conectores y estructura gramatical.
- `avg_word_len_mean vs long_word_ratio_mean`: es una pareja muy interpretable para identificar autores que no solo usan palabras más largas en promedio, sino que además recurren con frecuencia a vocabulario de mayor complejidad formal.
- `quote_count_per_1000_mean vs dialog_dash_count_per_1000_mean`: es probablemente el mejor par para detectar autores con fuerte presencia de diálogo, y además permite diferenciar convenciones de marcado del discurso directo.
- `quote_count_per_1000_mean vs question_count_per_1000_mean`: ayuda a identificar estilos más dialogados o interactivos, ya que el discurso directo suele venir acompañado de interrogación.
- `dialog_dash_count_per_1000_mean vs exclam_count_per_1000_mean`: puede separar autores con diálogo más expresivo o dramático de otros con diálogo más neutro o narrativo.
- `semicolon_count_per_1000_mean vs colon_count_per_1000_mean`: resulta muy útil para detectar preferencias de puntuación más marcadas y estilos con mayor tendencia a la articulación lógica o explicativa.
- `comma_count_per_1000_mean vs semicolon_count_per_1000_mean`: permite comparar dos estrategias distintas de complejidad sintáctica, separando autores que encadenan con comas de los que estructuran más con pausas fuertes.
- `punctuation_ratio_mean vs non_alpha_token_ratio_mean`: ayuda a captar estilos con mayor densidad gráfica o puntuacional, especialmente útil para separar escritura más limpia de escritura más marcada por signos.
- `long_sentence_ratio_mean vs long_word_ratio_mean`: es un buen par para ver si la complejidad sintáctica y la complejidad léxica van de la mano o si algunos autores destacan solo en una de las dos.
- `sentences_per_1000_tokens_mean vs avg_sentence_len_mean`: refleja una relación estructural muy clara, porque suele separar autores que fragmentan más el discurso de los que concentran más información en cada oración.

In [ ]:
scatter_pairs = [
    ("avg_sentence_len_mean", "comma_count_per_1000_mean"),
    ("avg_sentence_len_mean", "short_sentence_ratio_mean"),
    ("avg_sentence_len_mean", "long_sentence_ratio_mean"),
    ("avg_paragraph_len_mean", "avg_sentences_per_paragraph_mean"),
    ("mattr_100_mean", "avg_word_len_mean"),
    ("mattr_100_mean", "stopword_ratio_mean"),
    ("avg_word_len_mean", "long_word_ratio_mean"),
    ("quote_count_per_1000_mean", "dialog_dash_count_per_1000_mean"),
    ("quote_count_per_1000_mean", "question_count_per_1000_mean"),
    ("dialog_dash_count_per_1000_mean", "exclam_count_per_1000_mean"),
    ("semicolon_count_per_1000_mean", "colon_count_per_1000_mean"),
    ("comma_count_per_1000_mean", "semicolon_count_per_1000_mean"),
    ("punctuation_ratio_mean", "non_alpha_token_ratio_mean"),
    ("long_sentence_ratio_mean", "long_word_ratio_mean"),
    ("sentences_per_1000_tokens_mean", "avg_sentence_len_mean"),
]

In [ ]:
plot_scatter_list_plotly(data=desc_by_author, scatter_pairs=scatter_pairs, hue="author", work_col="author")

### Frecuencias de palabras por autor

Vamos a analizar ahora las frecuencias de palabras por autor para ver si se detectan patrones diferenciadores entre ellos. Para ello, vamos a calcular las frecuencias de las palabras alfabéticas más comunes en cada autor y luego vamos a visualizar estas frecuencias para comparar entre autores.

In [ ]:
# Obtenemos los tokens alfabéticos por autor
alpha_tokens_by_author = get_tokens_alpha_by_author(df_corpus)

In [ ]:
alpha_tokens_by_author.keys()

In [ ]:
author_counters = compute_word_frecuencies(alpha_tokens_by_author, STOPWORDS, remove_stopwords=False)
author_counters_nostop = compute_word_frecuencies(alpha_tokens_by_author, STOPWORDS, remove_stopwords=True)

In [ ]:
for autor, c in list(author_counters.items()):
    print(f"\n=== {autor} ===")
    print(top_n_words(c, 15))

In [ ]:
for autor, c in list(author_counters_nostop.items()):
    print(f"\n=== {autor} ===")
    print(top_n_words(c, 15))

### Top n-grams por autor

Vamos a analizar ahora los n-grams más frecuentes por autor para ver si se detectan patrones diferenciadores entre ellos. Para ello, vamos a calcular las frecuencias de los n-grams alfabéticos más comunes en cada autor y luego vamos a visualizar estas frecuencias para comparar entre autores.

In [ ]:
top_bigrams_by_author = compute_ngrams_frecuencies(alpha_tokens_by_author, stopword_set=STOPWORDS, ngram_n=2, top_k=20)
top_trigrams_by_author = compute_ngrams_frecuencies(alpha_tokens_by_author, stopword_set=STOPWORDS, ngram_n=3, top_k=20)

In [ ]:
top_bigrams_by_author

In [ ]:
top_trigrams_by_author

### Curvas de Zipf por autor

Las curvas de Zipf muestran una regularidad básica del lenguaje: unas pocas palabras aparecen con muchísima frecuencia y una gran mayoría aparecen muy poco. Cuando se ordenan las palabras por rango y se representa su frecuencia en escala log-log, suele observarse una relación aproximadamente lineal, lo que refleja esa estructura típica del vocabulario en textos escritos en lenguaje natural. En términos lingüísticos, la parte alta de la distribución está dominada por palabras funcionales muy frecuentes, mientras que la cola recoge palabras raras, específicas o de contexto.

En estilometría, comparar las curvas de Zipf entre autores sirve como una visión global de cómo se distribuye su léxico: permite observar si hay más concentración en palabras muy frecuentes, una cola léxica más larga o diferencias generales en repetición y variedad. Esto conecta con una idea central de la atribución de autoría: los patrones de frecuencia, especialmente de palabras funcionales, pueden reflejar hábitos estilísticos relativamente estables y menos dependientes del tema. Aun así, la utilidad de Zipf es sobre todo exploratoria y descriptiva, no concluyente por sí sola.

Además, estas curvas son útiles como herramienta de diagnóstico del corpus. Una curva anómala puede señalar no solo diferencias de estilo, sino también problemas de procesamiento o composición del corpus, como textos demasiado cortos, duplicados, errores de OCR o tokenización inconsistente. Por eso, las curvas de Zipf son valiosas en el análisis exploratorio inicial, pero para una estilometría más fina conviene complementarlas con medidas más discriminativas, como palabras frecuentes, n-gramas o distancias estilométricas específicas.

In [ ]:
plot_zipf_curve_by_author(alpha_tokens_by_author, FIGS_DIR / "zipf_by_author.png")

In [ ]:
plot_zipf_curve_by_author(alpha_tokens_by_author, FIGS_DIR / "zipf_by_author_max_rank_5000.png", max_rank=5000)